**NOTICE:**  
The U.S. Army Corps of Engineers, Risk Management Center (USACE-RMC) makes no guarantees about the results, or appropriateness of outputs, obtained from Numerics.

# 06. Optimization and Root Finding with Numerics
This notebook covers optimization and root finding methods in Numerics.

## What You'll Learn
- Root finding (Bisection, Brent's method)
- Numerical integration
- Local optimization (Nelder-Mead, BFGS, Powell)
- Global optimization (DE, MS, MLSL, Particle Swarm, SCE, Simulated Annealing)
- Practical applications

## When to Use Optimization vs MCMC

**Use Optimization when:**
- You need point estimates only (no uncertainty)
- Computational speed is critical
- You have a well-defined objective function

**Use MCMC when:**
- You need uncertainty quantification
- You want full posterior distributions
- You have prior information to incorporate

## Setup

In [ ]:
import pythonnet
pythonnet.load("coreclr")

import clr
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import time
import pandas as pd
from scipy import integrate
from matplotlib import animation
from IPython.display import HTML, display
import matplotlib as mpl
from System import Func, Double, Array
from System.Collections.Generic import List

# Load Numerics DLL
dll_path = Path(r"C:\GIT\Numerics\Numerics\bin\Debug\net8.0\Numerics.dll")
clr.AddReference(str(dll_path))

from Numerics.Mathematics.Optimization import (NelderMead, BFGS, Powell, ParticleSwarm, SimulatedAnnealing, AugmentedLagrange, 
DifferentialEvolution, MultiStart, LocalMethod, ShuffledComplexEvolution, MLSL, ConstraintType, IConstraint, Constraint)
from Numerics.Mathematics.RootFinding import Bisection, Brent
from Numerics.Mathematics.Integration import Integration

print("✓ Setup complete")

## 1. Root Finding

Root finding is the process of determining the values of $x$ for which a function $f(x) = 0$. These are also called zeros, roots, or solutions of the equation. Root finding is fundamental to many numerical methods and applications, including solving nonlinear equations, finding equilibrium points, and numerical integration of differential equations.

### Methods Available

| Method | Requires | Convergence | Best For |
|--------|----------|-------------|----------|
| **Bisection** | Bracketing interval | Linear (slow but sure) | Robust, guaranteed convergence |
| **Secant** | Two initial points | Superlinear (~1.618) | When derivative unavailable |
| **Newton-Raphson** | Derivative | Quadratic (very fast) | Smooth functions, good initial guess |
| **Brent** | Bracketing interval | Superlinear | General purpose, best overall |

### Problem Formulation

Given a function $f: \mathbb{R} \rightarrow \mathbb{R}$, find $x^*$ such that:

```math
f(x^*) = 0
```

### Bracketing

Some methods require a **bracketing interval** $[a, b]$ where $f(a)$ and $f(b)$ have opposite signs. By the Intermediate Value Theorem, if $f$ is continuous, there must be at least one root in the interval.

### Example: Quadratic Function

Find the root of $f(x) = x^2 - 2$ (true root: $\sqrt{2} \approx 1.414$)

We will explore the Bisection and Brent root finding method.


### Bisection Method

The bisection method is the simplest root-finding algorithm. It repeatedly bisects an interval and selects the subinterval in which the root must lie.

**Algorithm:**
1. Start with interval $[a, b]$ where $f(a) \cdot f(b) < 0$
2. Compute midpoint $c = (a + b) / 2$
3. If $f(c)$ is close enough to zero, return $c$
4. Otherwise, replace either $a$ or $b$ with $c$ based on the sign of $f(c)$
5. Repeat until convergence

**Convergence Rate:**
Bisection has **linear convergence** with a constant factor of $1/2$. After $n$ iterations, the error is bounded by:

```math
|e_n| \leq \frac{b - a}{2^n}
```

where $b - a$ is the initial interval width. This means each iteration gains approximately one binary digit of accuracy. To achieve a tolerance $\varepsilon$, the number of iterations required is:

```math
n \geq \frac{\log(b - a) - \log(\varepsilon)}{\log 2}
```

For example, starting with $[0, 3]$ and tolerance $10^{-10}$, bisection needs at most $\lceil \log_2(3 \times 10^{10}) \rceil = 35$ iterations. This predictability is one of bisection's strengths — you know exactly how many iterations you need before you start.

**Advantages:**
- Always converges if initial interval brackets a root
- Very robust - works even for discontinuous functions
- Simple to implement and understand
- Guaranteed to find a root

**Disadvantages:**
- Slow convergence (linear, each iteration reduces error by half)
- Requires bracketing interval
- Cannot find roots where function doesn't change sign (e.g., $f(x) = x^2$)

**When to use:** When robustness is paramount, or for poorly behaved functions.

### Brent's Method

Brent's method is the recommended general-purpose root-finding algorithm in the ***Numerics*** library. It combines the guaranteed convergence of bisection with the speed of the secant method and inverse quadratic interpolation, automatically switching between these strategies to achieve robust, fast convergence without requiring derivatives. For the vast majority of root-finding problems, Brent's method should be the first method you reach for.

**Mathematical Description:**
Brent's method maintains a bracketing interval $[a, b]$ such that $f(a)$ and $f(b)$ have opposite signs, guaranteeing that a root exists within the interval by the Intermediate Value Theorem. At each iteration, it selects one of three strategies to propose the next approximation:

**Bisection.** The simplest strategy: take the midpoint of the current bracket.

```math
x_{\text{bisect}} = \frac{a + b}{2}
```

Bisection reduces the bracket by exactly half at each step, giving linear convergence with error bound $|e_n| \leq (b - a) / 2^n$. It is slow but absolutely reliable.

**Secant method.** When only two distinct points are available (i.e., the previous contrapoint $a$ equals the older contrapoint $c$), the algorithm uses the secant formula, which fits a line through the two most recent function values and finds where it crosses zero:

```math
x_{\text{secant}} = b - f(b) \cdot \frac{b - a}{f(b) - f(a)}
```

The secant method converges superlinearly with order approximately $\varphi \approx 1.618$ (the golden ratio) for well-behaved functions.

**Inverse quadratic interpolation (IQI).** When three distinct points $a$, $b$, $c$ with distinct function values are available, the algorithm fits an inverse quadratic (a parabola through the three points with $x$ as a function of $y$) and evaluates it at $y = 0$:

```math
x_{\text{IQI}} = \frac{f_b f_c \cdot a}{(f_a - f_b)(f_a - f_c)} + \frac{f_a f_c \cdot b}{(f_b - f_a)(f_b - f_c)} + \frac{f_a f_b \cdot c}{(f_c - f_a)(f_c - f_b)}
```

where $f_a = f(a)$, $f_b = f(b)$, $f_c = f(c)$. IQI can converge even faster than the secant method when the function is smooth and the iterates are close to the root.

#### How It Works: The Decision Logic

The power of Brent's method lies in how it decides which strategy to use at each step. The algorithm follows a specific decision procedure that ensures it never loses the safety of bisection while taking faster steps whenever possible. Here is the logic as implemented in the ***Numerics*** library:

1. **Maintain the bracket.** The algorithm tracks three points: $b$ (the current best estimate, where $|f(b)|$ is smallest), $a$ (the previous iterate), and $c$ (the contrapoint such that $f(b)$ and $f(c)$ have opposite signs). If $f(b)$ and $f(c)$ have the same sign, the contrapoint is reset to $a$.

2. **Compute the midpoint and tolerance.** At each step, compute:

```math
x_m = \frac{c - b}{2}, \qquad \text{tol}_1 = 2 \varepsilon_{\text{mach}} |b| + \frac{\text{tol}}{2}
```

where $\varepsilon_{\text{mach}}$ is machine epsilon and $\text{tol}$ is the user-specified tolerance. If $|x_m| \leq \text{tol}_1$ or $f(b) = 0$, the root has been found.

3. **Try a fast step.** If the previous step $e$ was large enough ($|e| \geq \text{tol}_1$) and $b$ is improving ($|f(a)| > |f(b)|$), then attempt an open method:
   - If $a = c$ (only two distinct points available), use the **secant method**.
   - If $a \neq c$ (three distinct points available), use **inverse quadratic interpolation**.

4. **Accept or reject the fast step.** The proposed step $d = p/q$ is accepted only if it satisfies two conditions:
   - The step must be smaller than three-quarters of the distance to the midpoint: $2|p| < 3 |x_m \cdot q| - |\text{tol}_1 \cdot q|$
   - The step must be smaller than half the previous step: $2|p| < |e \cdot q|$

   These conditions ensure the method is making adequate progress toward the root. If either condition fails, the algorithm falls back to bisection.

5. **Update.** Apply the accepted step (or bisection fallback) to produce the new iterate $b$, evaluate $f(b)$, and repeat.

This design means that Brent's method takes fast steps when they are safe and productive, but always has bisection as a backstop, so it never diverges.

**Convergence Properties:**

Brent's method offers a rare combination of guaranteed convergence and fast practical performance:

- **Guaranteed convergence.** Like bisection, the method always maintains a valid bracket around the root. It will converge to a root for any continuous function where the initial interval satisfies $f(a) \cdot f(b) < 0$, regardless of how ill-conditioned the function is.

- **Superlinear convergence in practice.** For smooth, well-behaved functions, the algorithm typically achieves superlinear convergence by spending most iterations in secant or IQI mode. In the best case, IQI converges with order approximately 1.839.

- **Worst case is bisection.** When the function is poorly behaved (discontinuous derivatives, sharp curvature changes, or near-flat regions), the method gracefully degrades to bisection's linear convergence rate of $O(1/2^n)$. This is a floor, not a failure.

- **No pathological failure modes.** Unlike Newton-Raphson, which can cycle, diverge, or overshoot for bad initial guesses, and unlike the secant method, which can leave the bracket entirely, Brent's method has no failure modes beyond exceeding the maximum iteration count.

The convergence tolerance in the ***Numerics*** implementation uses a combined criterion: the root is accepted when the bracket half-width $|x_m|$ is within $\text{tol}_1 = 2\varepsilon_{\text{mach}}|b| + \text{tol}/2$, or when $f(b) = 0$ exactly. This accounts for both absolute and relative precision near the root.

#### Why Brent's Method Is the Recommended Default

For solving $f(x) = 0$ in a single variable, Brent's method should be the default choice because:

- **No derivatives needed.** Unlike Newton-Raphson, Brent's method requires only function evaluations. There is no need to derive, implement, or numerically approximate $f'(x)$, which is a significant practical advantage.

- **Guaranteed to converge.** Given a valid bracketing interval where $f(a)$ and $f(b)$ have opposite signs, the method will always find a root. Newton-Raphson and the secant method have no such guarantee.

- **Typically faster than bisection.** While bisection requires approximately $\log_2((b-a)/\varepsilon)$ iterations, Brent's method usually converges in far fewer iterations by exploiting the smoothness of the function. For the test function $f(x) = x^3 - 2x - 5$ on $[2, 3]$, bisection needs 27 iterations while Brent needs only 5.

- **No pathological failure cases.** Newton-Raphson can diverge, oscillate, or cycle when the initial guess is poor, the derivative is near zero, or the function has inflection points. Brent's method avoids all of these failure modes.

- **Handles both well-behaved and ill-conditioned functions.** The adaptive switching between IQI, secant, and bisection means the algorithm performs well across a wide range of function behaviors without requiring the user to diagnose the function's properties in advance.

### When NOT to Use Brent's Method

While Brent's method is the best general-purpose choice, there are situations where a different method is more appropriate:

- **Derivatives are available and the function is smooth.** If you already have an analytical derivative $f'(x)$ and the function is smooth with a good initial guess, Newton-Raphson will converge faster (quadratically vs. superlinearly). For functions where the derivative is cheap to evaluate, the `NewtonRaphson.RobustSolve` method provides Newton's speed with bisection's safety net.

- **Systems of equations or multiple dimensions.** Brent's method is strictly a univariate solver. For systems of nonlinear equations $\mathbf{F}(\mathbf{x}) = \mathbf{0}$, use multidimensional optimization methods such as Nelder-Mead or Newton-based nonlinear solvers.

- **The root is not bracketed.** Brent's method requires an initial interval $[a, b]$ where $f(a)$ and $f(b)$ have opposite signs. If you cannot identify such a bracket, consider using the `Brent.Bracket` helper method (see below), the secant method, or Newton-Raphson which do not require bracketing.

- **The function does not change sign at the root.** For roots of even multiplicity (e.g., $f(x) = x^2$ has a root at $x = 0$ but does not change sign), bracketing methods cannot be used. In these cases, reformulate the problem or use Newton-Raphson on $g(x) = f(x)/f'(x)$.

In [ ]:
# Define the function
def quadratic(x):
    """f(x) = x^2 - 2"""
    return x * x - 2.0

quadratic_func = Func[Double, Double](quadratic)
root = Bisection.Solve(quadratic_func, 1.0, 0.0, 4.0)
root_brent = Brent.Solve(quadratic_func, 0.0, 4.0)
true_root = np.sqrt(2.0)

root_df = pd.DataFrame([
    {'Method': 'Bisection', 'Found Root': root, 'True Root': true_root, 'Abs Error': abs(root - true_root), 'f(root)': quadratic(root)},
    {'Method': 'Brent', 'Found Root': root_brent, 'True Root': true_root, 'Abs Error': abs(root_brent - true_root), 'f(root)': quadratic(root_brent)},
])
print('Root-finding results for f(x) = x2 - 2')
display(root_df)

x = np.linspace(-0.5, 3, 500)
y = [quadratic(xi) for xi in x]
plt.figure(figsize=(10, 6))
plt.scatter(np.sqrt(2),0, color = 'red', label = 'Root', linewidths=5)
plt.plot(x, y, linewidth=2, label='f(x) = x2 - 2')
plt.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
plt.xlabel('x'); plt.ylabel('f(x)'); plt.title('Root Finding Example 1'); plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

### Performance Comparison: Numerics vs SciPy
We take a look at how fast Numerics root finding methods are against the Python implementation in the "scipy package". Here Numerics is a little slower, but it still a comparable speed.

In [ ]:
try:
    from scipy.optimize import root_scalar
    
    # Time Numerics
    start = time.perf_counter()
    root_numerics = Bisection.Solve(quadratic_func, 1, 0, 4)
    time_numerics = time.perf_counter() - start
    
    # Time SciPy
    start = time.perf_counter()
    result_scipy = root_scalar(quadratic, bracket=(1, 4), method='bisect', xtol=1e-8)
    root_scipy = result_scipy.root
    time_scipy = time.perf_counter() - start
    
    print("PERFORMANCE COMPARISON: Root Finding")
    print(f"Numerics Bisection:  {time_numerics*1000:.4f} ms  ->  root = {root_numerics:.10f}")
    print(f"SciPy Bisection:     {time_scipy*1000:.4f} ms  ->  root = {root_scipy:.10f}")
    print(f"Difference:          {abs(root_numerics - root_scipy):.2e}")
    
except ImportError:
    print("SciPy not installed. Skipping comparison.")

### Additional Root Finding Examples

Let's try more complex functions.

In [ ]:
# Trigonometric function: 2sin(x) - 3cos(x) - 0.5
def trig_func(x):
    return 2*np.sin(x) - 3*np.cos(x) - 0.5

trig_func_net = Func[Double, Double](trig_func)
root_trig = Bisection.Solve(trig_func_net, 0.5, 0.0, np.pi)
root_trig_brent = Brent.Solve(trig_func_net, 0.0, np.pi)
true_root_trig = 1.1219469809174553

def exp_func(x):
    return np.exp(-x) - x

exp_func_net = Func[Double, Double](exp_func)
root_exp = Bisection.Solve(exp_func_net, 1.0, -2.0, 2.0)
root_exp_brent = Brent.Solve(exp_func_net, -2.0, 2.0)
true_root_exp = 0.5671432904097838

results_df = pd.DataFrame([
    {'Function': '2sin(x)-3cos(x)-0.5', 'Method': 'Bisection', 
     'Root': root_trig, 
     'f(root)': trig_func(root_trig), 
     'Abs Error vs True': abs(root_trig - true_root_trig)},
    {'Function': '2sin(x)-3cos(x)-0.5', 'Method': 'Brent',    
     'Root': root_trig_brent, 
     'f(root)': trig_func(root_trig_brent), 
     'Abs Error vs True': abs(root_trig_brent - true_root_trig)},
    {'Function': 'e-x - x',           'Method': 'Bisection', 
     'Root': root_exp, 
     'f(root)': exp_func(root_exp), 
     'Abs Error vs True': abs(root_exp - true_root_exp)},
    {'Function': 'e-x - x',           'Method': 'Brent',    
     'Root': root_exp_brent, 
     'f(root)': exp_func(root_exp_brent), 
     'Abs Error vs True': abs(root_exp_brent - true_root_exp)},
])

print('Root-finding comparison table')
display(results_df)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
x1 = np.linspace(0, np.pi, 500)
ax1.plot(x1, [trig_func(xi) for xi in x1], linewidth=2, label = 'f(x) 2sin(x) - 3cos(x) - 0.5' )
ax1.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax1.scatter(true_root_trig, 0, color = 'red', linewidths=5, label='Root')
ax1.set_title('Root Finding Example 2'); ax1.legend(); ax1.grid(True, alpha=0.3)
ax1.legend()

x2 = np.linspace(-2, 2, 500)
ax2.plot(x2, [exp_func(xi) for xi in x2], linewidth=2, color='coral', label = 'f(x) = e-x - x')
ax2.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax2.scatter(true_root_exp, 0, color = 'red', linewidths=5, label = 'Root')
ax2.set_title('Root Finding Example 3'); ax2.legend(); ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout(); plt.show()

## 2. Numerical Integration

Compute definite integrals numerically when analytical integration is unavailable or inconvenient.

### Methods Available
- Gauss-Legendre quadrature: High accuracy for smooth functions with few evaluations.
- Trapezoidal rule: Simple baseline method with linear convergence.
- Simpson's rule: Higher-order rule using quadratic interpolation.
- Midpoint method: Simple and often slightly better than trapezoidal for smooth functions.

In [ ]:
def f_x3(x):
    return x**3

f_x3_net = Func[Double, Double](f_x3)

gauss = Integration.GaussLegendre(f_x3_net, 0, 1)
trapz = Integration.TrapezoidalRule(f_x3_net, 0, 1, 1000)
simps = Integration.SimpsonsRule(f_x3_net, 0, 1, 1000)
midpt = Integration.Midpoint(f_x3_net, 0, 1, 1000)

true_value = 0.25
int_df = pd.DataFrame([
    {'Method':'Gauss-Legendre','Result':gauss,'Abs Error':abs(gauss-true_value)},
    {'Method':'Trapezoidal','Result':trapz,'Abs Error':abs(trapz-true_value)},
    {'Method':"Simpson's Rule",'Result':simps,'Abs Error':abs(simps-true_value)},
    {'Method':'Midpoint','Result':midpt,'Abs Error':abs(midpt-true_value)},
])
print('Integration of x3 from 0 to 1')
display(int_df)

### Integration Performance: Numerics vs SciPy
We take a look at how fast Numerics integration methods are against Python's implementations in the "scipy" package. We can see that Numerics is a little slower, but the differences are comparable.

In [ ]:
try:
    start = time.perf_counter()
    result_numerics_gauss = Integration.GaussLegendre(f_x3_net, 0, 1)
    time_numerics_gauss = time.perf_counter() - start
    start = time.perf_counter()
    result_numerics_simps = Integration.SimpsonsRule(f_x3_net, 0, 1, 1000)
    time_numerics_simps = time.perf_counter() - start

    start = time.perf_counter()
    result_scipy_quad, _ = integrate.quad(f_x3, 0, 1)
    time_scipy_quad = time.perf_counter() - start
    x_pts = np.linspace(0, 1, 1001)
    y_pts = [f_x3(xi) for xi in x_pts]
    start = time.perf_counter()
    result_scipy_simps = integrate.simpson(y_pts, x_pts)
    time_scipy_simps = time.perf_counter() - start

    perf_df = pd.DataFrame([
        {'Family':'Gauss/Quad','Library':'Numerics','Method':'Gauss-Legendre','Time (ms)':time_numerics_gauss*1000,'Result':result_numerics_gauss},
        {'Family':'Gauss/Quad','Library':'SciPy','Method':'quad','Time (ms)':time_scipy_quad*1000,'Result':result_scipy_quad},
        {'Family':'Simpson','Library':'Numerics','Method':'Simpson','Time (ms)':time_numerics_simps*1000,'Result':result_numerics_simps},
        {'Family':'Simpson','Library':'SciPy','Method':'simpson','Time (ms)':time_scipy_simps*1000,'Result':result_scipy_simps},
    ])
    print('Performance comparison: integration (Numerics vs SciPy)')
    display(perf_df)
except ImportError:
    print('SciPy not installed. Skipping comparison.')

### Adaptive Integration
Adaptive integration refines evaluation intervals where local error is high and uses fewer evaluations where the integrand is smooth.

#### Methods
- Adaptive Gauss Lobatto
- Adaptive Gauss Kronrod
- Adaptive Simpson's Rule
- Adaptive Simpson's Rule 2D
- Vegas (Monte Carlo importance sampling)

In [ ]:
# Adaptive integration demo using a challenging function
def adaptive_target(x):
    return np.exp(-x*x)

adaptive_target_net = Func[Double, Double](adaptive_target)

# Reference value using SciPy quad
ref_val, _ = integrate.quad(adaptive_target, -3, 3)

# Compare fixed-grid methods at multiple resolutions (adaptive-like refinement study)
print(f"Reference integral value (SciPy quad): {ref_val:.10f}")
for n in [50, 100, 500, 1000, 5000]:
    rows = []
    trap = Integration.TrapezoidalRule(adaptive_target_net, -3, 3, n)
    simp = Integration.SimpsonsRule(adaptive_target_net, -3, 3, n)
    midp = Integration.Midpoint(adaptive_target_net, -3, 3, n)
    rows.extend([
        {'Method':'Trapezoidal','Subintervals':n,'Estimate':trap,'Abs Error':abs(trap-ref_val)},
        {'Method':'Simpson','Subintervals':n,'Estimate':simp,'Abs Error':abs(simp-ref_val)},
        {'Method':'Midpoint','Subintervals':n,'Estimate':midp,'Abs Error':abs(midp-ref_val)},
    ])

    adaptive_df = pd.DataFrame(rows)
    display(adaptive_df)

## Optimization
An optimization problem seeks to find:

```math
\min_{\mathbf{x}} f(\mathbf{x})
```

subject to:

```math
\mathbf{x}_L \leq \mathbf{x} \leq \mathbf{x}_U
```

where $f(\mathbf{x})$ is the objective function, $\mathbf{x}$ is the parameter vector, and $\mathbf{x}_L$, $\mathbf{x}_U$ are lower and upper bounds.

For maximization problems, minimize $-f(\mathbf{x})$ or use the `Maximize()` method.

## 3. Local Optimization

Local optimization methods find the nearest local minimum from a starting point. They are fast and efficient for smooth, unimodal functions but may get trapped in local minima for multimodal problems.

### Methods Available
- Adam
- BFGS
- Brent Search
- Golden Selection
- Gradient Descent
- Nelder Mead
- Powell

BFGS is gradient-based and requires derivatives, while Nelder-Mead and Powell are derivative-free. Gradient-based solvers converge fast when the objective is smooth and derivatives are reliable. Derivative-free solvers are more robust when gradients are noisy or unavailable.

#### BFGS (Broyden-Fletcher-Goldfarb-Shanno)
BFGS is a quasi-Newton method that builds an approximation to the Hessian matrix using gradient information [1]. It's one of the most effective methods for smooth, unconstrained optimization.

**Advantages:** Fast convergence, memory efficient, works well for most smooth problems.

**Disadvantages:** Can fail on non-smooth functions, requires bounded search space.

#### Nelder-Mead Simplex
The Nelder-Mead method is a direct search algorithm that uses a simplex (geometric figure with n+1 vertices in n dimensions) to search the parameter space. It's robust and doesn't require derivatives.

**Advantages:** Very robust, doesn't require derivatives, handles non-smooth functions.

**Disadvantages:** Slower convergence than gradient-based methods, can stagnate.

#### Powell's Method
Powell's method is a conjugate direction algorithm that doesn't require derivatives. It performs successive line searches along conjugate directions.

**Advantages:** No derivatives required, good for smooth functions.

**Disadvantages:** Can be slow in high dimensions, sensitive to scaling.

### Example: Rosenbrock Function
Rosenbrock is a classic optimization test: $f(x, y) = (a - x)^2 + b(y - x^2)^2$.
With a=1 and b=100, the global minimum is at (1,1) with f=0.

In [ ]:
# Rosenbrock function
def rosenbrock(params):
    n = len(params)
    F = 0
    for i in range(n-1):
        F += 100 * (params[i + 1] - params[i] * params[i])**2 + (1 - params[i])**2
    return F

# Visualize the function
x = np.linspace(-2, 2, 400)
y = np.linspace(-1, 3, 400)
X, Y = np.meshgrid(x, y)
Z = rosenbrock([X, Y])

fig = plt.figure(figsize=(14, 5))

# Contour plot
ax1 = fig.add_subplot(121)
contour = ax1.contour(X, Y, Z, levels=np.logspace(-1, 3.5, 20), cmap='viridis')
ax1.plot(1, 1, 'r*', markersize=20, label='Global Minimum (1, 1)')
ax1.set_xlabel('x', fontsize=12)
ax1.set_ylabel('y', fontsize=12)
ax1.set_title('Rosenbrock Function - Contour Plot', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 3D surface
ax2 = fig.add_subplot(122, projection='3d')
ax2.plot_surface(X, Y, Z, cmap='viridis', alpha=0.8, edgecolor='none')
ax2.scatter([1], [1], [0], color='red', s=100, label='Minimum')
ax2.set_xlabel('x', fontsize=10)
ax2.set_ylabel('y', fontsize=10)
ax2.set_zlabel('f(x, y)', fontsize=10)
ax2.set_title('Rosenbrock Function - 3D Surface', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("Global minimum:")
print(f"  Location: (1, 1)")
print(f"  Value:    {rosenbrock([1, 1])}")

# Initialize optimization parameters
initial = Array[Double]([0.0, 0.0])
lower = Array[Double]([-2.048, -2.048])
upper = Array[Double]([2.048, 2.048])

rosenbrock_net = Func[Array[Double], Double](rosenbrock)

# Optimization routines
bfgs = BFGS(rosenbrock_net, 2, initial, lower, upper)
bfgs.Minimize()
bfgs_soln = bfgs.BestParameterSet.Values

nelder = NelderMead(rosenbrock_net, 2, initial, lower, upper)
nelder.Minimize()
nelder_soln = nelder.BestParameterSet.Values

powell = Powell(rosenbrock_net, 2, initial, lower, upper)
powell.Minimize()
powell_soln = powell.BestParameterSet.Values

print('\n'  + " Finding Minimum of Rosenbrock Function:")

table = pd.DataFrame([
    {'Method': 'BFGS',
                'Result': np.round(bfgs_soln, 9),
                'Error': np.round(list(bfgs_soln) - np.array([1, 1]), 9)},
        {'Method': 'Nelder-Mead',
                'Result': np.round(nelder_soln,9),
                'Error': np.round(list(nelder_soln) - np.array([1, 1]), 9)},
        {'Method': 'Powell',
                'Result': np.round(powell_soln, 14),
                'Error': np.round(list(powell_soln) - np.array([1, 1]), 14)}])

display(table)


Now we will look at the path the solver takes when trying to find the minimum of the Rosenbrock function!

In [ ]:
# Added: use Numerics trace output for BFGS path and objective values
bfgs_trace = BFGS(rosenbrock_net, 2, initial, lower, upper)
bfgs_trace.RecordTraces = True
bfgs_trace.Minimize()

# Extract trace from Numerics
trace_list = list(bfgs_trace.ParameterSetTrace)
trace_params = np.array([[float(ps.Values[0]), float(ps.Values[1])] for ps in trace_list], dtype=float)
trace_values = np.array([float(ps.Fitness) for ps in trace_list], dtype=float)

# Make sure the best parameter set is included in the path
best_params = np.array([float(bfgs_trace.BestParameterSet.Values[0]), float(bfgs_trace.BestParameterSet.Values[1])], dtype=float)
if len(trace_params) == 0 or not np.allclose(trace_params[-1], best_params):
    trace_params = np.vstack([trace_params, best_params])
    trace_values = np.append(trace_values, float(bfgs_trace.BestParameterSet.Fitness))

# Plots!
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(trace_values, color='steelblue', linewidth=1)
axes[0].set_title('BFGS Objective Trace', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Objective value')
axes[0].grid(True, alpha=0.3)

if len(trace_params) > 1:
    axes[1].contour(X, Y, Z, levels=np.logspace(-1, 3.5, 20), cmap='viridis')
    axes[1].plot(trace_params[:, 0], trace_params[:, 1], color='red', linewidth=1.5, label='BFGS path')
    axes[1].scatter([trace_params[0, 0]], [trace_params[0, 1]], color='white', edgecolor='black', s=50, zorder=5, label='Start')
    # Zorder=10 brings the end point in front of the contour lines
    axes[1].scatter([best_params[0]], [best_params[1]], color='yellow', edgecolor='black', s=50, zorder=10, label='End')
    axes[1].set_title('BFGS Path on Rosenbrock Contours', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('x')
    axes[1].set_ylabel('y')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)


# Animated path
try:
    from matplotlib import animation
    from IPython.display import HTML, display
    import matplotlib as mpl
    mpl.rcParams['animation.embed_limit'] = 50  # MB - raise ceiling to 50 MB to see full animation

    if len(trace_params) > 1:
        # Increased hold_frames from 20 to 80 so the final minimum is clearly visible
        hold_frames = 80
        anim_path = np.vstack([trace_params, np.repeat([trace_params[-1]], hold_frames, axis=0)])
        fig, ax = plt.subplots(figsize=(6, 5))
        ax.contour(X, Y, Z, levels=np.logspace(-1, 3.5, 20), cmap='viridis')
        line, = ax.plot([], [], color='red', linewidth=1.5)
        point, = ax.plot([], [], 'ro', markersize=4)
        # Static start/end markers so they are visible throughout the animation
        ax.scatter([trace_params[0, 0]], [trace_params[0, 1]], color='white', edgecolor='black', s=50, zorder=5, label='Start')
        ax.scatter([best_params[0]], [best_params[1]], color='yellow', edgecolor='black', s=50, zorder=10, label='End')
        ax.set_title('BFGS Path Animation', fontsize=12, fontweight='bold')
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        ax.legend()
        ax.grid(True, alpha=0.3)
        def init():
            line.set_data([], [])
            point.set_data([], [])
            return line, point
        def update(frame):
            xs = anim_path[:frame + 1, 0]
            ys = anim_path[:frame + 1, 1]
            line.set_data(xs, ys)
            point.set_data([anim_path[frame, 0]], [anim_path[frame, 1]])
            return line, point
        anim = animation.FuncAnimation(fig, update, init_func=init, frames=len(anim_path), interval=50, blit=True, repeat=False)
        display(HTML(anim.to_jshtml()))
        plt.close(fig)
except Exception as e:
    print('Inline animation rendering skipped:', e)


### Example: McCormick Function
Another classic optimization function is the McCormick function. With $f(x, y) = \sin(x + y) + (x - y)^2 - 1.5x + 2.5y + 1$. This has a minimum of -1.9133 at f(-0.54719, -1.54719).

In [ ]:
def mccormick(params):
    x, y = params[0], params[1]
    return np.sin(x + y) + (x - y)**2 - 1.5*x + 2.5*y + 1

# Visualize McCormick function
x = np.linspace(-3, 4, 400)
y = np.linspace(-3, 4, 400)
X, Y = np.meshgrid(x, y)
Z = mccormick([X, Y])

fig = plt.figure(figsize=(14, 5))
ax1 = fig.add_subplot(121)
contour = ax1.contour(X, Y, Z, levels=30, cmap='viridis')
ax1.plot(-0.54719, -1.54719, 'r*', markersize=20, label='Global Minimum (-0.54719, -1.54719)')
ax1.set_xlabel('x', fontsize=12)
ax1.set_ylabel('y', fontsize=12)
ax1.set_title('McCormick Function - Contour Plot', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2 = fig.add_subplot(122, projection='3d')
ax2.plot_surface(X, Y, Z, cmap='viridis', alpha=0.8, edgecolor='none')
ax2.set_xlabel('x', fontsize=10)
ax2.set_ylabel('y', fontsize=10)
ax2.set_zlabel('f(x, y)', fontsize=10)
ax2.set_title('McCormick Function - 3D Surface', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

# Initialize optimization parameters
initial = Array[Double]([0.0, 0.0])
lower = Array[Double]([-3.0, -3.0])
upper = Array[Double]([4.0, 4.0])

mccormick_net = Func[Array[Double], Double](mccormick)

# Optimization routines
bfgs = BFGS(mccormick_net, 2, initial, lower, upper)
bfgs.Minimize()
bfgs_soln = bfgs.BestParameterSet.Values

nelder = NelderMead(mccormick_net, 2, initial, lower, upper)
nelder.Minimize()
nelder_soln = nelder.BestParameterSet.Values

powell = Powell(mccormick_net, 2, initial, lower, upper)
powell.Minimize()
powell_soln = powell.BestParameterSet.Values

print('\n' + ' Finding Minimum of McCormick Function:')

table = pd.DataFrame([
    {'Method': 'BFGS',
                'Result': np.round(bfgs_soln, 9),
                'Error': np.round(list(bfgs_soln) - np.array([ -0.547, -1.547 ]), 9)},
        {'Method': 'Nelder-Mead',
                'Result': np.round(nelder_soln, 9),
                'Error': np.round(list(nelder_soln) - np.array([ -0.547, -1.547 ]), 9)},
        {'Method': 'Powell',
                'Result': np.round(powell_soln, 9),
                'Error': np.round(list(powell_soln) - np.array([ -0.547, -1.547 ]), 9)}])

display(table)

## 4. Global Optimization
Global optimization methods are designed to find the global minimum across the entire search space, avoiding local minima. They typically require more function evaluations but are more robust for multimodal problems.

### Methods Available
- Differential Evolution
- Multi-Start
- Multi-Start Single Linkage (MLSL)
- Particle Swarm
- Shuffled Complex Evolution
- Simulated Annealing

#### Differential Evolution

Differential Evolution (DE) is a population-based evolutionary algorithm that's very robust for continuous optimization [[4]](#4). For each member $\mathbf{x}_i$ of the population, DE creates a **trial vector** $\mathbf{u}$ using mutation and crossover:

**Mutation** (DE/rand/1): Three distinct population members $\mathbf{x}_{r_0}$, $\mathbf{x}_{r_1}$, $\mathbf{x}_{r_2}$ are randomly selected, and a **mutant vector** is formed:

```math
\mathbf{v} = \mathbf{x}_{r_0} + G \cdot (\mathbf{x}_{r_1} - \mathbf{x}_{r_2})
```

where $G$ is a scale factor. In the ***Numerics*** implementation, $G$ is **dithered** with 90% probability: $G = 0.5 + \text{rand}() \times 0.5$, which helps avoid stagnation.

**Crossover** (binomial): The trial vector $\mathbf{u}$ is assembled component-by-component:

```math
u_j = \begin{cases} v_j & \text{if } \text{rand}() \leq CR \text{ or } j = j_{\text{rand}} \\ x_{i,j} & \text{otherwise} \end{cases}
```

where $CR$ is the crossover probability and $j_{\text{rand}}$ is a randomly chosen index that ensures at least one component comes from the mutant vector.

**Selection** (greedy): The trial vector replaces the target only if it has equal or better fitness:

```math
\mathbf{x}_i^{(g+1)} = \begin{cases} \mathbf{u} & \text{if } f(\mathbf{u}) \leq f(\mathbf{x}_i^{(g)}) \\ \mathbf{x}_i^{(g)} & \text{otherwise} \end{cases}
```

The algorithm converges when the standard deviation of fitness values across the population falls below the tolerance.

**Advantages**: Very robust, handles discontinuities, good for difficult problems.

**Parameters**:
- `PopulationSize`: Number of candidate solutions (default = 10 × dimensions)
- `CrossoverProbability`: Probability of crossover (default = 0.9)
- `Mutation`: Scaling factor for mutation (default = 0.75)

#### Particle Swarm Optimization

PSO simulates social behavior of bird flocking or fish schooling [[5]](#5). Each particle $i$ has a position $\mathbf{x}_i$ and velocity $\mathbf{v}_i$ that are updated at each iteration using:

```math
v_{i,j}^{(t+1)} = w \cdot v_{i,j}^{(t)} + c_1 \cdot r_1 \cdot (p_{i,j} - x_{i,j}^{(t)}) + c_2 \cdot r_2 \cdot (g_j - x_{i,j}^{(t)})
```
```math
x_{i,j}^{(t+1)} = x_{i,j}^{(t)} + v_{i,j}^{(t+1)}
```

where:
- $w$ is the **inertia weight**, which decreases linearly from $w_{\max} = 0.9$ to $w_{\min} = 0.4$ over the optimization, balancing exploration (high $w$) and exploitation (low $w$)
- $c_1 = c_2 = 2.05$ are the cognitive and social acceleration coefficients
- $r_1, r_2 \sim U(0,1)$ are independent random numbers (per component, per particle)
- $\mathbf{p}_i$ is particle $i$'s personal best position (best position it has visited)
- $\mathbf{g}$ is the global best position (best position any particle has visited)

The three terms represent **momentum** (continue in the current direction), **cognitive pull** (return toward personal best), and **social pull** (move toward the swarm's best).
**Advantages**: Fast convergence, simple concept, works well for continuous problems.

**Parameters**:
- `PopulationSize`: Number of particles (default = 30)

#### Shuffled Complex Evolution (SCE-UA)

SCE-UA was specifically developed for calibrating hydrological models [[6]](#6). The algorithm partitions the population into $p$ **complexes**, evolves each complex independently using a **Competitive Complex Evolution** (CCE) strategy, then shuffles members between complexes to share information.

The CCE step within each complex proceeds as follows:

1. **Sub-complex selection**: Select $q = D+1$ points from the complex using a trapezoidal probability distribution that favors better-ranked points: $P(i) = \frac{2(N+1-i)}{N(N+1)}$

2. **Reflection**: Compute the centroid $\mathbf{g}$ of all sub-complex points except the worst point $\mathbf{x}_w$, then reflect:

```math
\mathbf{r} = 2\mathbf{g} - \mathbf{x}_w
```

3. **Contraction**: If the reflected point is infeasible or worse than $\mathbf{x}_w$, try contraction:

```math
\mathbf{c} = \frac{\mathbf{g} + \mathbf{x}_w}{2}
```

4. **Mutation**: If contraction also fails, generate a random point within the smallest bounding box of the complex.

The shuffling step redistributes points across complexes, preventing any single complex from stagnating. This combination of local competitive evolution within complexes and global information sharing between complexes makes SCE-UA particularly effective for the rugged, multimodal objective functions typical of hydrological model calibration.

**Advantages**: Excellent for hydrological model calibration, balances exploration and exploitation.

**Best for**: Calibrating watershed models, water resources applications.

#### Simulated Annealing

SA mimics the physical process of annealing in metallurgy [[7]](#7). It accepts uphill moves with decreasing probability, allowing escape from local minima.
**Advantages**: Can escape local minima, works for discrete and continuous problems.

**Parameters**:
- `InitialTemperature`: Starting temperature (default = 10)
- `CoolingRate`: Temperature reduction factor (default = 0.95)

#### Multi-Start Optimization

Combines local search with multiple random starting points.

**Advantages**: Simple to implement, leverages fast local search.

**Disadvantages**: May waste evaluations in same basin of attraction.

#### MLSL (Multi-Level Single Linkage)

Clustering-based global optimization that avoids redundant local searches.

**Advantages**: More efficient than multi-start, avoids redundant searches.

### Example: 5D Rosenbrock Function
Below we apply these to a 5D Rosenbrock objective.

In [ ]:
# 5D objective visualized via 2D slice at fixed remaining coordinates.
lower = Array[Double]([-2.048, -2.048, -2.048, -2.048, -2.048])
upper = Array[Double]([2.048, 2.048, 2.048, 2.048, 2.048])

# Optimize with particle swarm
particle = ParticleSwarm(rosenbrock_net, 5, lower, upper)
particle.PopulationSize = 100
particle.MaxIterations = 100000
particle.Minimize()
particle_soln = particle.BestParameterSet.Values

# Optimize with simulated annealing
anneling = SimulatedAnnealing(rosenbrock_net, 5, lower, upper)
anneling.Minimize()
anneling_soln = anneling.BestParameterSet.Values

# Optimize with differential evolution
de = DifferentialEvolution(rosenbrock_net, 5, lower, upper)
de.PopulationSize = 100
de.MaxIterations = 5000
de.Minimize()
de_soln = de.BestParameterSet.Values

# Optimize with MultiStart
ms_initial = Array[Double]([0.0, 0.0, 0.0, 0.0, 0.0]) # Requires an intial guess with the same number of dimensions as the problem
ms = MultiStart(rosenbrock_net, 5, ms_initial, lower, upper, LocalMethod.BFGS) # Requires a local method to use for the local optimization steps
ms.MaxIterations = 50  # number of local starts
ms.Minimize()
ms_soln = ms.BestParameterSet.Values

# Optimize with MLSL
mlsl = MLSL(rosenbrock_net, 5, ms_initial, lower, upper, LocalMethod.BFGS) # Requires an itital guess with the same number of dimensions as the problem
mlsl.SampleSize = 30
mlsl.MaxIterations = 200
mlsl.Minimize()
mlsl_soln = mlsl.BestParameterSet.Values

# Optimize with Shuffled Complex Evolution
sce = ShuffledComplexEvolution(rosenbrock_net, 5, lower, upper)
sce.Complexes = 5
sce.MaxIterations = 2000
sce.Minimize()
sce_soln = sce.BestParameterSet.Values

print('\n'  + " Finding Minimum of Rosenbrock Function:")

table = pd.DataFrame([
    {'Method': 'Particle Swarm',
                'Result': np.round(particle_soln, 14),
                'Error': np.round(list(particle_soln) - np.array([1, 1, 1, 1, 1]), 14)},
        {'Method': 'Simulated Annealing',
                'Result': np.round(anneling_soln, 4),
                'Error': np.round(list(anneling_soln) - np.array([1, 1, 1, 1, 1]), 4)},
        {'Method': 'Differential Evolution',
                'Result': np.round(de_soln, 14),
                'Error': np.round(list(de_soln) - np.array([1, 1, 1, 1, 1]), 14)},
        {'Method': 'MultiStart',
                'Result': np.round(ms_soln, 14),
                'Error': np.round(list(ms_soln) - np.array([1, 1, 1, 1, 1]), 14)},
        {'Method': 'MLSL',
                'Result': np.round(mlsl_soln, 14),
                'Error': np.round(list(mlsl_soln) - np.array([1, 1, 1, 1, 1]), 14)},
        {'Method': 'Shuffled Complex Evolution',
                'Result': np.round(sce_soln, 14),
                'Error': np.round(list(sce_soln) - np.array([1, 1, 1, 1, 1]), 14)}
                ])

display(table)


### Examples: McCormick, Eggholder, Ackley, Three Hump Camel
We continue with the McCormick example from the local optimization section. We also introduce other test functions for optimization.

Eggholder: $f(x, y) = -(y + 47)\sin\left(\sqrt{|x/2 + (y + 47)|}\right) - x \cdot \sin\left(\sqrt{|x - (y + 47)|}\right)$ with a minimum of -959.6407 at $f(512, 404.2319)$        
Ackley: $f(x, y) = -20 \cdot \exp[-0.2\sqrt{0.5(x^2 + y^2)}] - \exp[0.5(\cos(2\pi x) + \cos(2\pi y))] + e + 20$ with a minimum of 0 at $f(0, 0)$        
Three Hump Camel: $f(x, y) = 2x^2 - 1.05x^4 + x^6/6 + xy + y^2$ with a minimum of 0 at $f(0, 0)$

In [ ]:
# Added: Global optimization tests on additional benchmark functions
def egghold(params):
    x, y = params[0], params[1]
    return -(y + 47) * np.sin(np.sqrt(abs(x/2 + (y + 47)))) - x * np.sin(np.sqrt(abs(x - (y + 47))))

def ackley(params):
    x, y = params[0], params[1]
    return -20 * np.exp(-0.2 * np.sqrt(0.5 * (x*x + y*y))) - np.exp(0.5 * (np.cos(2*np.pi*x) + np.cos(2*np.pi*y))) + np.e + 20

def three_hump_camel(params):
    x, y = params[0], params[1]
    return 2*x*x - 1.05*x**4 + (x**6)/6 + x*y + y*y

def run_global_benchmark(func, name, lower_b, upper_b, min_x, min_y):
    # Visualize
    x = np.linspace(lower_b, upper_b, 400)
    y = np.linspace(lower_b, upper_b, 400)
    X, Y = np.meshgrid(x, y)
    Z = func([X, Y])

    fig = plt.figure(figsize=(14, 5))
    ax1 = fig.add_subplot(121)
    contour = ax1.contour(X, Y, Z, levels=30, cmap='viridis')
    ax1.plot(min_x, min_y, 'r*', markersize=20, label=f'Global Minimum ({min_x}, {min_y})')
    ax1.set_xlabel('x', fontsize=12)
    ax1.set_ylabel('y', fontsize=12)
    ax1.set_title(f'{name} - Contour Plot', fontsize=12, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2 = fig.add_subplot(122, projection='3d')
    ax2.plot_surface(X, Y, Z, cmap='viridis', alpha=0.8, edgecolor='none')
    ax2.set_xlabel('x', fontsize=10)
    ax2.set_ylabel('y', fontsize=10)
    ax2.set_zlabel('f(x, y)', fontsize=10)
    ax2.set_title(f'{name} - 3D Surface', fontsize=12, fontweight='bold')

    plt.tight_layout()
    plt.show()

    lower = Array[Double]([lower_b, lower_b])
    upper = Array[Double]([upper_b, upper_b])

    func_net = Func[Array[Double], Double](func)

    # Optimize with particle swarm
    particle = ParticleSwarm(func_net, 2, lower, upper)
    particle.PopulationSize = 100
    particle.MaxIterations = 100000
    particle.Minimize()
    particle_soln = particle.BestParameterSet.Values

    # Optimize with simulated annealing
    anneling = SimulatedAnnealing(func_net, 2, lower, upper)
    anneling.Minimize()
    anneling_soln = anneling.BestParameterSet.Values

    # Optimize with differential evolution
    de = DifferentialEvolution(func_net, 2, lower, upper)
    de.PopulationSize = 100
    de.MaxIterations = 5000
    de.Minimize()
    de_soln = de.BestParameterSet.Values

    # Optimize with MultiStart
    ms_initial = Array[Double]([0.0, 0.0])
    ms = MultiStart(func_net, 2, ms_initial, lower, upper, LocalMethod.BFGS)
    ms.MaxIterations = 50
    ms.Minimize()
    ms_soln = ms.BestParameterSet.Values

    # Optimize with MLSL
    mlsl = MLSL(func_net, 2, ms_initial, lower, upper, LocalMethod.BFGS)
    mlsl.SampleSize = 30
    mlsl.MaxIterations = 200
    mlsl.Minimize()
    mlsl_soln = mlsl.BestParameterSet.Values

    # Optimize with Shuffled Complex Evolution
    sce = ShuffledComplexEvolution(func_net, 2, lower, upper)
    sce.Complexes = 5
    sce.MaxIterations = 2000
    sce.Minimize()
    sce_soln = sce.BestParameterSet.Values

    print('\n' + f' Finding Minimum of {name}:')
    table = pd.DataFrame([
        {'Method': 'Particle Swarm',
                    'Result': np.round(particle_soln, 6)},
            {'Method': 'Simulated Annealing',
                    'Result': np.round(anneling_soln, 6)},
            {'Method': 'Differential Evolution',
                    'Result': np.round(de_soln, 6)},
            {'Method': 'MultiStart',
                    'Result': np.round(ms_soln, 6)},
            {'Method': 'MLSL',
                    'Result': np.round(mlsl_soln, 6)},
            {'Method': 'Shuffled Complex Evolution',
                    'Result': np.round(sce_soln, 6)}
                    ])
    display(table)

# Run global benchmarks
run_global_benchmark(mccormick, 'McCormick Function', -3.0, 4.0, -0.54719, -1.54719)
run_global_benchmark(egghold, 'Egghold Function', -512.0, 512.0, 512, 404.2319)
run_global_benchmark(ackley, 'Ackley Function', -5.0, 5.0, 0, 0)
run_global_benchmark(three_hump_camel, 'Three Hump Camel Function', -5.0, 5.0, 0, 0)


Sticking with the Eggholder function we will take a look at the path the Differential Evolition solver takes when trying to find the minimum!

In [ ]:
eggholder_net = Func[Array[Double], Double](egghold)
lower = Array[Double]([-512, -512])
upper = Array[Double]([512, 512])

# Contour grid
x_eg = np.linspace(-512, 512, 400)
y_eg = np.linspace(-512, 512, 400)
X_eg, Y_eg = np.meshgrid(x_eg, y_eg)
Z_eg = egghold([X_eg, Y_eg])

# Run DE with trace recording
de_trace = DifferentialEvolution(eggholder_net, 2, lower, upper)
de_trace.PopulationSize = 100
de_trace.MaxIterations = 5000
de_trace.RecordTraces = True
de_trace.Minimize()

# Extract trace
trace_list = list(de_trace.ParameterSetTrace)
trace_params = np.array([[float(ps.Values[0]), float(ps.Values[1])] for ps in trace_list], dtype=float)
trace_values = np.array([float(ps.Fitness) for ps in trace_list], dtype=float)

# Ensure best parameter set is included at the end
best_params = np.array([float(de_trace.BestParameterSet.Values[0]), float(de_trace.BestParameterSet.Values[1])], dtype=float)
if len(trace_params) == 0 or not np.allclose(trace_params[-1], best_params):
    trace_params = np.vstack([trace_params, best_params])
    trace_values = np.append(trace_values, float(de_trace.BestParameterSet.Fitness))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Objective trace
axes[0].plot(trace_values, color='steelblue', linewidth=1)
axes[0].set_title('DE Objective Trace (Eggholder)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Objective value')
axes[0].grid(True, alpha=0.3)

# Contour with path
if len(trace_params) > 1:
    axes[1].contour(X_eg, Y_eg, Z_eg, levels=30, cmap='viridis')
    axes[1].plot(trace_params[:, 0], trace_params[:, 1], color='red', linewidth=2, alpha=0.7, label='DE path')
    axes[1].scatter([trace_params[0, 0]], [trace_params[0, 1]], color='white', edgecolor='black', s=50, zorder=5, label='Start')
    axes[1].scatter([best_params[0]], [best_params[1]], color='yellow', edgecolor='black', s=50, zorder=10, label='End')
    axes[1].set_title('DE Path on Eggholder Contours', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('x')
    axes[1].set_ylabel('y')
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Animation
try:
    mpl.rcParams['animation.embed_limit'] = 80  # raise ceiling to 80 MB

    if len(trace_params) > 1:
        hold_frames = 80
        anim_path = np.vstack([trace_params, np.repeat([trace_params[-1]], hold_frames, axis=0)])
        # Skip every other frame
        anim_path = anim_path[::2]

        fig, ax = plt.subplots(figsize=(6, 5), dpi=72)
        ax.contour(X_eg, Y_eg, Z_eg, levels=30, cmap='viridis')
        line, = ax.plot([], [], color='red', linewidth=2)
        point, = ax.plot([], [], 'ro', markersize=4)

        # Static markers visible throughout
        ax.scatter([trace_params[0, 0]], [trace_params[0, 1]], color='white', edgecolor='black', s=50, zorder=5, label='Start')
        ax.scatter([best_params[0]], [best_params[1]], color='yellow', edgecolor='black', s=50, zorder=10, label='End')

        ax.set_title('DE Path Animation (Eggholder)', fontsize=12, fontweight='bold')
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

        def init():
            line.set_data([], [])
            point.set_data([], [])
            return line, point

        def update(frame):
            xs = anim_path[:frame + 1, 0]
            ys = anim_path[:frame + 1, 1]
            line.set_data(xs, ys)
            point.set_data([anim_path[frame, 0]], [anim_path[frame, 1]])
            return line, point

        anim = animation.FuncAnimation(fig, update, init_func=init, frames=len(anim_path),
                                        interval=50, blit=True, repeat=False)
        display(HTML(anim.to_jshtml()))
        plt.close(fig)

except Exception as e:
    print('Inline animation rendering skipped:', e)

## 5. Constrained Optimization (Augmented Lagrange)

We will now use the Rosenbrock function constrained to a disk to test the Augmented Lagrange Algorithm, which handles equality and inequality constraints by adding penalty terms to the objective function. This method uses an inner solver; here it is BFGS, which is gradient-based. The constraint handling is separate from the inner solver, so you can swap in derivative-free solvers when needed.

**Constraint Types**:
- `ConstraintType.EqualTo`: Equality constraint $g(\mathbf{x}) = 0$
- `ConstraintType.LesserThanOrEqualTo`: Inequality constraint $g(\mathbf{x}) \leq 0$
- `ConstraintType.GreaterThanOrEqualTo`: Inequality constraint $g(\mathbf{x}) \geq 0$


In [ ]:
def constraint(x):
    return x[0]*x[0] + x[1]*x[1]

def rosenbrock_disk(params):
    return np.power(1 - params[0], 2) + 100*(np.power(params[1] - params[0]*params[0], 2))

rosenbrock_disk_net = Func[Array[Double], Double](rosenbrock_disk)
constraint_net = Func[Array[Double], Double](constraint)

# Set up inner solver
initial = Array[Double]([0.0, 0.0])
lower = Array[Double]([-1.5, -1.5])
upper = Array[Double]([1.5, 1.5])
inner_solver = BFGS(rosenbrock_disk_net, 2, initial, lower, upper)

# Set up constraint
constraint = Constraint(constraint_net, 2, 2, ConstraintType.LesserThanOrEqualTo)

# Solve
constraint_list = List[IConstraint]()
constraint_list.Add(constraint)

solver = AugmentedLagrange(rosenbrock_disk_net, inner_solver, constraint_list)
solver.Minimize()
soln = solver.BestParameterSet.Values

# Results
print('\n'  + " Finding Minimum of Rosenbrock Function Constrained to a Disk:")

table = pd.DataFrame([
    {'Method': 'Augmented Lagrange',
                'Result': np.round(soln, 6),
                'Error': np.round(list(soln) - np.array([1, 1]), 6)},
])
display(table)


## Summary

### When to Use Each Method
Table includes root finding, integration, local, global, and constrained optimization methods covered above.
|Category|Method|Best Use|
|---|---|---|
|Root Finding | Bisection| Robust, bracketed roots|
|Root Finding | Brent | Faster than bisection, bracketed roots |
|Integration | Gauss-Legendre | High accuracy for smooth functions |
|Integration | Simpson | Good accuracy/speed tradeoff|
|Local Optimization | BFGS | Smooth objectives, fast convergence|
|Local Optimization| Nelder-Mead | Noisy/derivative free |
|Global Optimization | Particle Swarm | Multimodal, easy tuning |
|Global Optimization| DE | Robust, global search |
|Global Optimization | SCE | Hydrolofic calibration |
|Constrained | Augmented Lagrange | Explicit constraints |

## Practical Tips
- Start with broad bounds to avoid excluding the true optimum.
- Use multiple starts for non-convex objectives.
- For noisy objectives, prefer derivative-free methods (Nelder-Mead, Powell).
- Always verify solutions by evaluating the objective value at the returned parameters.

## Next Steps
- **07_statistics.ipynb**: Hypothesis testing and transformations.
- **05_mcmc_advanced.ipynb**: When uncertainty quantification is needed.

## Exercise
1. Fit Rosenbrock in 3D with BFGS and DE.
2. Compare runtime and final error.
3. Try narrower bounds and observe convergence.

